# Replay: backtest and live

`replay` merges several stored streams in index order and yields their events one at a time, optionally sleeping between them. The `speed` argument controls the pace: `float("inf")` means no sleeping, so events arrive as fast as the CPU allows, which is the mode for offline backtesting. A finite `speed` inserts a wait of `index_gap / speed` between events, giving downstream code realistic timing for a live-like run.

The events, their values and their order, are identical at every speed; only the gaps change. A strategy developed and tested at max speed runs unchanged at live speed.

In [ ]:
import numpy as np
import pandas as pd
from screamer import replay

# two stored streams; the index holds logical timestamps (e.g. seconds)
a_val = np.array([1.0, 2.0, 3.0])
a_idx = np.array([0, 10, 30], dtype=np.int64)
b_val = np.array([0.5, 2.5])
b_idx = np.array([5, 20], dtype=np.int64)


async def drain(events):
    "Collect every event from an async generator into a list."
    out = []
    async for e in events:
        out.append(e)
    return out

## Backtest: max speed

`replay` is `Merge` (notebook 07) plus optional pacing. With `speed=float("inf")`
there is no sleeping: the merged, index-sorted events come out as fast as
possible. Each event is a `(value, index, source)` triple, where `source` says
which input it came from.

In [ ]:
backtest = await drain(replay(a_val, b_val, index=[a_idx, b_idx], speed=float("inf")))

pd.DataFrame({
    "index":  [i for v, i, s in backtest],
    "value":  [v for v, i, s in backtest],
    "source": [("a" if s == 0 else "b") for v, i, s in backtest],
})

## Live: wall-clock replay

With a finite `speed`, `replay` waits between events by `index_gap / speed`. In
production that wait is a real `asyncio.sleep`; here we inject a fake sleep that
records the requested delay without blocking, so the demo is instant and
deterministic while still exercising the timing.

In [ ]:
slept = []

async def fake_sleep(seconds):
    "Record the requested wait instead of actually sleeping."
    slept.append(seconds)

live = await drain(replay(a_val, b_val, index=[a_idx, b_idx], speed=2.0, sleep=fake_sleep))

print("injected waits (index_gap / speed):", slept)

## Same events, only the timing differs

The events are identical to the backtest; only the injected waits are new. The
table shows the shared `(index, value, source)` and the wait inserted before each
event in replay (the first event has no preceding wait). A strategy that consumed
the backtest stream sees exactly this sequence live.

In [ ]:
same_events = [(v, i, s) for v, i, s in backtest] == [(v, i, s) for v, i, s in live]
print("backtest and replay yield the same events:", same_events)

pd.DataFrame({
    "index":  [i for v, i, s in live],
    "value":  [v for v, i, s in live],
    "source": [("a" if s == 0 else "b") for v, i, s in live],
    "wait before (replay)": [0.0] + slept,
})